<a href="https://colab.research.google.com/github/Ashritha0848/NLP_Assignments/blob/main/2403A52229%2CBatch_9%2CAssignment_12_2%2CNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:

import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

from collections import Counter

# ==========================================================
# STEP 3 — Create Dataset (Manual Sentences)
# ==========================================================

sentences = [
    "Win money now",
    "Call this number to claim prize",
    "Congratulations you won a lottery",
    "Free entry in contest now",
    "Claim your free gift card",

    "Hey how are you",
    "Lets meet tomorrow",
    "Are you coming to class",
    "Call me when you are free",
    "Good morning have a nice day"
]

# Labels: 1 = spam, 0 = ham
labels = [1,1,1,1,1, 0,0,0,0,0]

print("Sample sentences:", sentences[:3])
print("Labels:", labels[:3])

# ==========================================================
# STEP 4 — Text Preprocessing
# ==========================================================

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text

cleaned = [clean_text(s) for s in sentences]

def tokenize(text):
    return text.split()

tokens = [tokenize(s) for s in cleaned]

# ==========================================================
# STEP 5 — Vocabulary and Encoding
# ==========================================================

all_words = [word for sentence in tokens for word in sentence]
word_counts = Counter(all_words)

vocab = {word: idx+1 for idx, (word, _) in enumerate(word_counts.items())}

def encode(sentence):
    return [vocab[word] for word in sentence]

encoded = [encode(s) for s in tokens]

# Padding
MAX_LEN = 10

def pad(seq):
    if len(seq) < MAX_LEN:
        seq += [0]*(MAX_LEN - len(seq))
    else:
        seq = seq[:MAX_LEN]
    return seq

padded = [pad(seq) for seq in encoded]

# ==========================================================
# STEP 6 — Train-Test Split
# ==========================================================

X = np.array(padded)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.long)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# ==========================================================
# STEP 7 — Multi-Filter CNN Model
# ==========================================================

class TextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super(TextCNN, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        # Multiple filters
        self.conv1 = nn.Conv1d(embed_dim, 100, kernel_size=3)
        self.conv2 = nn.Conv1d(embed_dim, 100, kernel_size=4)
        self.conv3 = nn.Conv1d(embed_dim, 100, kernel_size=5)

        self.relu = nn.ReLU()

        self.fc = nn.Linear(300, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)
        x = x.permute(0, 2, 1)

        c1 = self.relu(self.conv1(x))
        c2 = self.relu(self.conv2(x))
        c3 = self.relu(self.conv3(x))

        p1 = torch.max(c1, dim=2)[0]
        p2 = torch.max(c2, dim=2)[0]
        p3 = torch.max(c3, dim=2)[0]

        out = torch.cat((p1, p2, p3), dim=1)

        out = self.fc(out)
        return self.sigmoid(out).squeeze()

# Initialize model
vocab_size = len(vocab) + 1
model = TextCNN(vocab_size, 50)

# ==========================================================
# STEP 8 — Training
# ==========================================================

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()

    optimizer.zero_grad()
    outputs = model(X_train)

    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

# ==========================================================
# STEP 9 — Evaluation
# ==========================================================

model.eval()
with torch.no_grad():
    preds = model(X_test)
    preds = (preds > 0.5).float()

y_pred = preds.numpy()
y_true = y_test.numpy()

print("\nAccuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# ==========================================================
# END
# ==========================================================

Sample sentences: ['Win money now', 'Call this number to claim prize', 'Congratulations you won a lottery']
Labels: [1, 1, 1]
Epoch 1, Loss: 0.6834568381309509
Epoch 2, Loss: 0.513388454914093
Epoch 3, Loss: 0.38921648263931274
Epoch 4, Loss: 0.2946406900882721
Epoch 5, Loss: 0.22350060939788818

Accuracy: 0.5
Precision: 0.0
Recall: 0.0
F1 Score: 0.0
Confusion Matrix:
 [[1 0]
 [1 0]]


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
